In [1]:
import os, zipfile, glob

os.makedirs("../data", exist_ok=True)
base_dir = "../input/dogs-vs-cats-redux-kernels-edition"

with zipfile.ZipFile(os.path.join(base_dir, "train.zip")) as z:
    z.extractall("../data")
with zipfile.ZipFile(os.path.join(base_dir, "test.zip")) as z:
    z.extractall("../data")

extracted_root = os.path.join("../data", "dogs-vs-cats-redux-kernels-edition")
train_dir = os.path.join(extracted_root, "train")
test_dir = os.path.join(extracted_root, "test")
print(f"train_dir: {train_dir}\ntest_dir: {test_dir}")



train_dir: ../data/dogs-vs-cats-redux-kernels-edition/train
test_dir: ../data/dogs-vs-cats-redux-kernels-edition/test


In [2]:
train_list = glob.glob(os.path.join(train_dir, "**", "*.jpg"), recursive=True)
test_list = glob.glob(os.path.join(test_dir, "**", "*.jpg"), recursive=True)

from sklearn.model_selection import train_test_split

train_list, val_list = train_test_split(train_list, test_size=0.1, random_state=42)

print(
    f"found {len(train_list)} training, {len(val_list)} validation, {len(test_list)} test images"
)



found 20250 training, 2250 validation, 2500 test images


In [3]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128



In [4]:
from torchvision import transforms


class ImageTransform:
    def __init__(self):
        self.data_transform = {
            "train": transforms.Compose(
                [
                    transforms.RandomResizedCrop(224),
                    transforms.RandomHorizontalFlip(),
                    transforms.ToTensor(),
                    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
                ]
            ),
            "val": transforms.Compose(
                [
                    transforms.Resize(250),
                    transforms.CenterCrop(224),
                    transforms.ToTensor(),
                    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
                ]
            ),
            "test": transforms.Compose(
                [
                    transforms.Resize(250),
                    transforms.CenterCrop(224),
                    transforms.ToTensor(),
                    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
                ]
            ),
        }

    def __call__(self, img, phase):
        return self.data_transform[phase](img)




In [5]:
from torch.utils import data
from PIL import Image


class MyDataset(data.Dataset):
    def __init__(self, file_path_list, transform, phase):
        self.file_list = file_path_list
        self.transform = transform
        self.phase = phase

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path).convert("RGB")
        img_transformed = self.transform(img, self.phase)

        if self.phase in ("train", "val"):
            label_str = os.path.basename(img_path).split(".")[0]
            label = 1 if label_str == "dog" else 0
            return img_transformed, label
        else:  # test
            file_id = os.path.splitext(os.path.basename(img_path))[0]
            return img_transformed, file_id




In [6]:
transform = ImageTransform()

train_dataset = MyDataset(train_list, transform=transform, phase="train")
train_dataloader = data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)

val_dataset = MyDataset(val_list, transform=transform, phase="val")
val_dataloader = data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)

test_dataset = MyDataset(test_list, transform=transform, phase="test")
test_dataloader = data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)




In [7]:
import torch.nn as nn


class Cnn(nn.Module):
    def __init__(self):
        super(Cnn, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, stride=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(3 * 3 * 64, 10)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(10, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = out.view(out.size(0), -1)
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out




In [8]:
model = Cnn()
model.train()




Cnn(
  (layer1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Linear(in_features=576, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=10, out_features=2, bias=True)
  (re

In [9]:
from tqdm import tqdm
import torch.nn.functional as F
import copy


def train_model(
    model, train_loader, val_loader, optimizer, criterion, epochs, scheduler=None
):
    """
    Train for the given number of epochs, keep the weights that achieve the
    lowest validation loss, and return the best‑model.
    """
    model = model.to(DEVICE)

    best_val_loss = float("inf")
    best_state = None

    def _train(epoch):
        model.train()
        epoch_loss = 0.0
        epoch_acc = 0.0
        for data, label in tqdm(
            train_loader, desc=f"Training Epoch {epoch+1}", leave=False
        ):
            data = data.to(DEVICE)
            label = label.to(DEVICE)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()
            acc = (output.argmax(dim=1) == label).float().mean()
            epoch_loss += loss.item()
            epoch_acc += acc.item()
        print(
            f"Epoch {epoch+1} Train Loss: {epoch_loss/len(train_loader):.4f} "
            f"Acc: {epoch_acc/len(train_loader):.4f}"
        )

    def _val(epoch):
        model.eval()
        val_loss = 0.0
        val_acc = 0.0
        with torch.no_grad():
            for data, label in tqdm(
                val_loader, desc=f"Validation Epoch {epoch+1}", leave=False
            ):
                data = data.to(DEVICE)
                label = label.to(DEVICE)
                output = model(data)
                loss = criterion(output, label)
                acc = (output.argmax(dim=1) == label).float().mean()
                val_loss += loss.item()
                val_acc += acc.item()
        avg_val_loss = val_loss / len(val_loader)
        print(
            f"Epoch {epoch+1} Val   Loss: {avg_val_loss:.4f} "
            f"Acc: {val_acc/len(val_loader):.4f}"
        )
        return avg_val_loss

    for epoch in range(epochs):
        _train(epoch)
        cur_val_loss = _val(epoch)
        if cur_val_loss < best_val_loss:
            best_val_loss = cur_val_loss
            best_state = copy.deepcopy(model.state_dict())
        if scheduler is not None:
            scheduler.step()

    # Load the best weights before returning
    if best_state is not None:
        model.load_state_dict(best_state)
    return model




In [10]:
import torch.optim as optim

optimizer = optim.Adam(
    model.parameters(), lr=5e-4, weight_decay=1e-4
)  # stronger weight decay
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.StepLR(
    optimizer, step_size=3, gamma=0.5
)  # smoother decay

model = train_model(
    model,
    train_dataloader,
    val_dataloader,
    optimizer,
    criterion,
    epochs=12,  # train a few more epochs
    scheduler=scheduler,
)




Epoch 1 Train Loss: 0.6296 Acc: 0.6397


Epoch 1 Val   Loss: 0.5890 Acc: 0.6711


Epoch 2 Train Loss: 0.5604 Acc: 0.7079


Epoch 2 Val   Loss: 0.5405 Acc: 0.7275


Epoch 3 Train Loss: 0.5430 Acc: 0.7228


...

Epoch 10 Val   Loss: 0.3985 Acc: 0.8218


Epoch 11 Train Loss: 0.4550 Acc: 0.7842


Epoch 11 Val   Loss: 0.3906 Acc: 0.8278


Epoch 12 Train Loss: 0.4601 Acc: 0.7798


Epoch 12 Val   Loss: 0.3960 Acc: 0.8207


In [11]:
model.eval()
dog_probs = []
with torch.no_grad():
    for data, file_id in tqdm(test_dataloader, desc="Predict Test"):
        data = data.to(DEVICE)
        logits = model(data)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        dog_probs.extend(zip(file_id, probs))

dog_probs.sort(key=lambda x: int(x[0]))
ids, probs = zip(*dog_probs)




Predict Test: 100%|██████████| 20/20 [00:02<00:00,  8.48it/s]


In [12]:
import pandas as pd

submission = pd.DataFrame({"id": ids, "label": probs})
submission.head()




,id,label
0,1,0.995057
1,2,0.165038
2,3,0.203856
3,4,0.675049
4,5,0.355631


In [13]:
submission.to_csv("result.csv", index=False)
print("Saved submission to result.csv")

Saved submission to result.csv
